In [2]:
import torch
import torch.nn as nn
from torch import optim as optim
from torch.nn import functional as F
import torch.utils.data as data
import math
import copy
from dataclasses import dataclass

In [3]:
@dataclass
class Config:
  d_model: int = 512
  n_heads: int = 8
  n_layers: int = 6
  d_ff: int = 2048
  dropout: int = 0.1
  src_vocab_size: int = 0
  tgt_vocab_size: int = 0
  T: int = 512

In [4]:
class PositionalEncoding(nn.Module):
  def __init__(self, config):
    super().__init__()
    T, d_model = config.T, config.d_model
    pe = torch.zeros(T, d_model) # (T, d_model)
    position = torch.arange(0, T, dtype=torch.float).unsqueeze(1) # (T, 1)
    i = torch.arange(0, d_model, 2) # (d_model//2,)
    div_term = 1 / (10000 ** (i/d_model)) # (d_model//2,)
    pe[:, 0::2] = torch.sin(position * div_term) # (T, d_model//2,)
    pe[:, 1::2] = torch.cos(position * div_term) # (T, d_model//2,)
    self.register_buffer("pe", pe.unsqueeze(0)) # (1, T, d_model//2,)

  def forward(self, x):
    B, T, C = x.size()
    return x + self.pe[:, :T, :]

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.n_heads = config.n_heads
    self.head_size = self.d_model // self.n_heads
    self.W_q = nn.Linear(self.d_model, self.d_model)
    self.W_k = nn.Linear(self.d_model, self.d_model)
    self.W_v = nn.Linear(self.d_model, self.d_model)
    self.W_o = nn.Linear(self.d_model, self.d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    # Q, K, V: (B, n_heads, T, head_size)
    attn_scores = Q @ K.transpose(-1, -2) # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) => (B, n_heads, T, T)
    attn_scores = attn_scores / math.sqrt(self.head_size)
    if mask is not None:
      attn_scores = attn_scores.masked_fill(mask==0, float("-inf"))
    attn_probs = torch.softmax(attn_scores, dim=-1)
    output = attn_probs @ V # (B, n_heads, T, T) @ (B, n_heads, T, head_size) => (B, n_heads, T, head_size)
    return output

  def split_heads(self, x):
    B, T, C = x.size() # (B, T, d_model)
    return x.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, head_size)

  def combine_heads(self, x):
    B, n_heads, T, head_size = x.size() # (B, n_heads, T, head_size)
    return x.transpose(1, 2).view(B, T, n_heads*head_size) # (B, T, head_size)

  def forward(self, Q, K, V, mask=None):
    Q = self.split_heads(self.W_q(Q)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    K = self.split_heads(self.W_k(K)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    V = self.split_heads(self.W_v(V)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    attn_output = self.scaled_dot_product(Q, K, V, mask)
    output = self.W_o(self.combine_heads(attn_output)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model)
    return output

In [6]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.fc1 = nn.Linear(self.d_model, 4*self.d_model)
    self.fc2 = nn.Linear(4*self.d_model, self.d_model)
    self.relu = nn.ReLU()

  def forward(self, x):
    return self.fc2(self.relu(self.fc1(x))) # (B, T, d_model)

In [7]:
class EncoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.self_attn = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
    self.norm1 = nn.LayerNorm(config.d_model)
    self.norm2 = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, mask):
    attn_output = self.self_attn(x, x, x, mask) # (B, T, d_model)
    x = self.norm1(x + self.dropout(attn_output)) # (B, T, d_model)
    ff_output = self.feed_forward(x) # (B, T, d_model)
    x = self.norm2(x + self.dropout(ff_output)) # (B, T, d_model)
    return x

In [8]:
class DecoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.self_attn = MultiHeadAttention(config)
    self.cross_attn = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
    self.norm1 = nn.LayerNorm(config.d_model)
    self.norm2 = nn.LayerNorm(config.d_model)
    self.norm3 = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, enc_output, src_mask, tgt_mask):
    attn_output = self.self_attn(x, x, x, tgt_mask) # (B, T, d_model)
    x = self.norm1(x + self.dropout(attn_output)) # (B, T, d_model)

    cross_attn_output = self.cross_attn(x, enc_output, enc_output, src_mask) # (B, T, d_model)
    x = self.norm2(x + self.dropout(cross_attn_output)) # (B, T, d_model)

    ff_output = self.feed_forward(x) # (B, T, d_model)
    x = self.norm3(x + self.dropout(ff_output)) # (B, T, d_model)
    return x

In [ ]:
class Transformer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.dropout = config.dropout
    self.n_layers = config.n_layers

    self.encoder_embedding = nn.Embedding(config.src_vocab_size, self.d_model)
    self.decoder_embedding = nn.Embedding(config.tgt_vocab_size, self.d_model)
    self.positional_encoding = PositionalEncoding(config)

    self.encoder_layers = nn.ModuleList(EncoderLayer(config) for _ in range(self.n_layers))
    self.decoder_layers = nn.ModuleList(DecoderLayer(config) for _ in range(self.n_layers))

    self.fc = nn.Linear(self.d_model, config.tgt_vocab_size)
    self.dropout = nn.Dropout(self.dropout)

  def generate_mask(self, src, tgt):
    # src (B, src_tokens)
    # tgt (B, tgt_tokens)

    # padding masks
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2) # (B, 1, 1, src_tokens)
    tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(2) # (B, 1, 1, tgt_tokens)

    seq_length = tgt.size(1) # tgt_tokens

    no_peak_mask = torch.tril(
        torch.ones(seq_length, seq_length, dtype=torch.bool, device=tgt.device)
    )

    tgt_mask = tgt_mask & no_peak_mask
    return src_mask, tgt_mask

  def forward(self, src, tgt):
    # src : (B, src_len)
    # tgt : (B, tgt_len)
    src_mask, tgt_mask = self.generate_mask(src, tgt)
    src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src))) # (B, src_len) -> (B, src_len, d_model)
    tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt))) # (B, src_len) -> (B, src_len, d_model)

    enc_output = src_embedded
    for enc_layer in self.encoder_layers:
      enc_output = enc_layer(enc_output, src_mask) # (B, src_len, d_model)

    dec_output = tgt_embedded
    for dec_layer in self.decoder_layers:
      dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask) # (B, tgt_len, d_model)

    output = self.fc(dec_output) # (B, tgt_len, tgt_vocab_size)
    return output